In [1]:
# ==============================================================================
# CELL 1: KHỞI TẠO MÔI TRƯỜNG & KẾT NỐI LƯU TRỮ ĐÁM MÂY
# Mục đích: Nhập các thư viện cần thiết và thiết lập quyền truy cập Google Drive.
# ==============================================================================
from google.colab import drive
import os
import re
import shutil
from pathlib import Path

drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# ==============================================================================
# CELL 2: CẤU HÌNH ĐƯỜNG DẪN HỆ THỐNG
# Mục đích: Định nghĩa các hằng số môi trường (Environment Constants)
# Lưu ý: Cập nhật đường dẫn tuyệt đối theo cấu trúc thư mục thực tế trên Drive.
# ==============================================================================

INPUT_ROOT_DIR = '/content/drive/MyDrive/Camera_Data/Grouped'
OUTPUT_ROOT_DIR = '/content/drive/MyDrive/Camera_Data/Grouped_by_time'
UNIDENTIFIED_DIR_NAME = 'UNIDENTIFIED_CAMS'

# Thiết lập biểu thức chính quy (Regular Expression) để trích xuất thông tin.
# Phân tích định dạng tệp:
# Nhóm 1: Dấu thời gian (Timestamp) - dao động từ 14 đến 18 chữ số.
# Nhóm 2: Định danh thiết bị (Camera ID) - tùy chọn, từ 5 đến 15 ký tự chữ và số.
FILE_PATTERN = re.compile(r"^(\d{14,18})(?:_([A-Za-z0-9]{5,15}))?")

print(f"[CONFIG] Thư mục nguồn (Input): {INPUT_ROOT_DIR}")
print(f"[CONFIG] Thư mục đích (Output): {OUTPUT_ROOT_DIR}")

[CONFIG] Thư mục nguồn (Input): /content/drive/MyDrive/Camera_Data/Grouped
[CONFIG] Thư mục đích (Output): /content/drive/MyDrive/Camera_Data/Grouped_by_time


In [8]:
# ==============================================================================
# CELL 3: TIẾN TRÌNH PHÂN CỤM DỮ LIỆU THEO TÍNH LIÊN TỤC THỜI GIAN (TEMPORAL CONTINUITY CLUSTERING)
# Phân tầng 1: Định danh thiết bị vật lý (Camera ID)
# Phân tầng 2: Phân hoạch theo chu kỳ ngày (Temporal Partitioning by Date)
# Phân tầng 3: Phân hoạch theo phiên quan sát/ca học (Session/Continuity Clustering)
# Thuật toán áp dụng: Interval Merging kết hợp ngưỡng dung sai thời gian (Tolerance Window = 5s)
# Mục đích: Đảm bảo tính toàn vẹn của chuỗi thời gian (Time-series integrity) cho Phase 1.
# ==============================================================================

import cv2
import shutil
from datetime import datetime, timedelta
from pathlib import Path

def get_video_duration(file_path):
    """
    Trích xuất độ dài tệp tin video (Duration) thông qua API của OpenCV.
    Phương pháp: Phân tích Header thay vì giải mã toàn bộ khung hình (Frame Decoding)
    nhằm tối ưu hóa chi phí thao tác I/O (I/O Cost) trên hệ thống máy chủ.
    Output: Tổng thời lượng tính bằng giây (float).
    """
    cap = cv2.VideoCapture(str(file_path))
    if not cap.isOpened():
        return 0.0

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    if fps > 0:
        return frame_count / fps
    return 0.0

def organize_dataset_by_continuity(input_dir, output_dir, pattern, unidentified_name, overlap_tolerance_sec=5):
    print("[PROCESS] Khởi chạy thuật toán phân cụm dữ liệu dựa trên tính liên tục của chuỗi thời gian.")

    input_path = Path(input_dir)
    output_path = Path(output_dir)

    video_files = list(input_path.rglob("*.mov")) + list(input_path.rglob("*.MOV")) + \
                  list(input_path.rglob("*.mp4")) + list(input_path.rglob("*.MP4"))

    if not video_files:
        print("[ERROR] Không phát hiện tệp dữ liệu khả dụng. Quá trình xử lý bị hủy bỏ.")
        return

    print(f"[INFO] Tổng quy mô tập dữ liệu đầu vào: {len(video_files)} tệp. Đang tiến hành phân tích Metadata...")

    # Giai đoạn 1: Khởi tạo Cấu trúc Từ điển (Dictionary) để gom nhóm sơ bộ
    cam_dict = {}

    for file_path in video_files:
        match = pattern.search(file_path.name)
        if not match:
            continue # Loại bỏ các tệp không tuân thủ định dạng hệ thống

        timestamp_str = match.group(1)
        cam_id = match.group(2)

        if not cam_id:
            continue # Bỏ qua các tệp khuyết định danh thiết bị phần cứng

        # Chuyển đổi chuỗi định dạng sang đối tượng Datetime
        try:
            start_time = datetime.strptime(timestamp_str[:14], "%Y%m%d%H%M%S")
        except ValueError:
            continue

        date_str = start_time.strftime("%Y-%m-%d")

        if cam_id not in cam_dict:
            cam_dict[cam_id] = {}
        if date_str not in cam_dict[cam_id]:
            cam_dict[cam_id][date_str] = []

        cam_dict[cam_id][date_str].append({
            "path": file_path,
            "start_time": start_time,
            "name": file_path.name
        })

    # Giai đoạn 2: Khôi phục tính liên tục (Continuity) áp dụng Interval Merging
    tolerance = timedelta(seconds=overlap_tolerance_sec)
    success_count = 0

    for cam_id, dates in cam_dict.items():
        for date_str, files in dates.items():
            # Sắp xếp các phân đoạn theo trình tự thời gian tuyến tính
            files.sort(key=lambda x: x["start_time"])

            groups = []
            current_group = []
            current_end_time = None

            for f_info in files:
                duration_sec = get_video_duration(f_info["path"])
                f_info["duration"] = timedelta(seconds=duration_sec)

                start_t = f_info["start_time"]
                end_t = start_t + f_info["duration"]

                # Đánh giá tính liên kết vật lý của các phân đoạn video
                if not current_group:
                    current_group.append(f_info)
                    current_end_time = end_t
                else:
                    # Đạt điều kiện liên tục (Nằm trong khoảng dung sai cho phép)
                    if start_t <= (current_end_time + tolerance):
                        current_group.append(f_info)
                        # Cập nhật mốc thời gian kết thúc của toàn bộ chuỗi
                        current_end_time = max(current_end_time, end_t)
                    else:
                        # Ghi nhận đứt gãy cấu trúc thời gian (Structural Break) -> Đóng phiên quan sát
                        groups.append((current_group, current_end_time))
                        current_group = [f_info]
                        current_end_time = end_t

            if current_group:
                groups.append((current_group, current_end_time))

            # Giai đoạn 3: Tái cấu trúc không gian lưu trữ vật lý
            for grp, grp_end in groups:
                grp_start = grp[0]["start_time"]

                # Tổng hợp thời lượng thực tế của phiên quan sát (Session Duration)
                total_duration = int((grp_end - grp_start).total_seconds())
                mins, secs = divmod(total_duration, 60)

                folder_name = f"{grp_start.strftime('%H%M%S')}_Total_{mins}m{secs}s"

                target_dir = output_path / f"CAM_{cam_id}" / date_str / folder_name
                target_dir.mkdir(parents=True, exist_ok=True)

                for f_info in grp:
                    new_file_path = target_dir / f_info["name"]
                    try:
                        if not new_file_path.exists():
                            shutil.move(str(f_info["path"]), str(new_file_path))
                            success_count += 1
                    except Exception as e:
                        print(f"[ERROR] Quá trình dịch chuyển tệp {f_info['name']} thất bại. Chi tiết ngoại lệ: {e}")

    print("\n" + "="*85)
    print("[INFO] BÁO CÁO TỔNG KẾT TIẾN TRÌNH PHÂN CỤM THEO PHIÊN QUAN SÁT (SESSION CLUSTERING):")
    print(f" - Tổng số phân đoạn dữ liệu được định tuyến và ghép nối thành công: {success_count}")
    print(f" - Tham số cấu hình: Ngưỡng dung sai đứt gãy (Tolerance Window) = {overlap_tolerance_sec} giây")
    print(" - Kiến trúc lưu trữ đầu ra: CAM_ID / YYYY-MM-DD / [Giờ_Bắt_Đầu]_[Tổng_Thời_Lượng] / [Tệp_Dữ_Liệu]")
    print("="*85)

# Thực thi hàm với ngưỡng dung sai cấu hình (5 giây) để xử lý Overlap
organize_dataset_by_continuity(INPUT_ROOT_DIR, OUTPUT_ROOT_DIR, FILE_PATTERN, UNIDENTIFIED_DIR_NAME, overlap_tolerance_sec=5)

[PROCESS] Khởi chạy thuật toán phân cụm dữ liệu dựa trên tính liên tục của chuỗi thời gian.
[INFO] Tổng quy mô tập dữ liệu đầu vào: 271 tệp. Đang tiến hành phân tích Metadata...

[INFO] BÁO CÁO TỔNG KẾT TIẾN TRÌNH PHÂN CỤM THEO PHIÊN QUAN SÁT (SESSION CLUSTERING):
 - Tổng số phân đoạn dữ liệu được định tuyến và ghép nối thành công: 209
 - Tham số cấu hình: Ngưỡng dung sai đứt gãy (Tolerance Window) = 5 giây
 - Kiến trúc lưu trữ đầu ra: CAM_ID / YYYY-MM-DD / [Giờ_Bắt_Đầu]_[Tổng_Thời_Lượng] / [Tệp_Dữ_Liệu]


In [9]:
# ==============================================================================
# CELL 4: TRÍCH XUẤT PHÂN ĐOẠN DỮ LIỆU ĐẠT CHUẨN THỐNG KÊ (STATISTICAL SIGNIFICANCE FILTERING)
# Mục đích: Lọc và trích xuất các phiên quan sát (Sessions) có thời lượng đủ lớn (> 15 phút).
# Lý do: Đảm bảo chuỗi thời gian (Time-series) có đủ độ dài và phương sai (Variance)
# để thuật toán Causal Discovery (Phase 2) có thể hội tụ và phát hiện mối quan hệ nhân quả.
# ==============================================================================

import re
from pathlib import Path

# Cấu hình ngưỡng thời gian tối thiểu (Đơn vị: Phút)
MINIMUM_DURATION_MINUTES = 15.0

def filter_valid_sessions(output_dir, min_minutes):
    print(f"[PROCESS] Khởi chạy bộ lọc dữ liệu: Tìm kiếm các phiên quan sát >= {min_minutes} phút.")
    base_path = Path(output_dir)

    if not base_path.exists():
        print("[ERROR] Thư mục gốc không tồn tại. Vui lòng kiểm tra lại quá trình phân cụm ở Cell 3.")
        return []

    # Biểu thức chính quy trích xuất số phút và số giây từ cấu trúc tên thư mục
    # Ví dụ: Folder "085540_Total_45m30s" -> group(1) = 45, group(2) = 30
    duration_pattern = re.compile(r"_Total_(\d+)m(\d+)s")

    valid_sessions = []
    total_sessions_scanned = 0

    # Quét đệ quy tìm các thư mục phiên quan sát (chứa từ khóa "_Total_")
    print("[PROCESS] Đang quét cấu trúc cây thư mục và phân tích Metadata...")
    for folder_path in base_path.rglob("*_Total_*"):
        if folder_path.is_dir():
            total_sessions_scanned += 1
            match = duration_pattern.search(folder_path.name)

            if match:
                mins = int(match.group(1))
                secs = int(match.group(2))

                # Quy đổi tổng thời gian ra đơn vị phút (float)
                total_time_in_mins = mins + (secs / 60.0)

                # Áp dụng bộ lọc ngưỡng (Threshold Filtering)
                if total_time_in_mins >= min_minutes:
                    valid_sessions.append({
                        "path": folder_path,
                        "duration": total_time_in_mins,
                        "cam_id": folder_path.parent.parent.name, # Lấy tên CAM_XXX
                        "date": folder_path.parent.name           # Lấy YYYY-MM-DD
                    })

    # Sắp xếp các phiên quan sát đạt chuẩn theo thời lượng giảm dần (Ưu tiên dữ liệu dài nhất)
    valid_sessions.sort(key=lambda x: x["duration"], reverse=True)

    # Xuất báo cáo thống kê
    print("\n" + "="*80)
    print("[INFO] BÁO CÁO KẾT QUẢ SÀNG LỌC DỮ LIỆU ĐẦU VÀO CHO PHASE 1:")
    print(f" - Tổng số phiên quan sát đã quét: {total_sessions_scanned}")
    print(f" - Số lượng phiên đạt tiêu chuẩn (>= {min_minutes} phút): {len(valid_sessions)}")
    print("="*80)

    if valid_sessions:
        print("[INFO] DANH SÁCH DỮ LIỆU ƯU TIÊN (Sắp xếp theo thời lượng giảm dần):")
        for idx, session in enumerate(valid_sessions, 1):
            # Cắt bớt phần path dài dòng của Colab để báo cáo nhìn gọn gàng
            short_path = f"{session['cam_id']}/{session['date']}/{session['path'].name}"
            print(f"  {idx:02d}. [{session['duration']:.2f} phút] -> {short_path}")
    else:
        print("[WARNING] Không có phiên quan sát nào đạt ngưỡng thời gian yêu cầu.")

    return valid_sessions

# Thực thi Pipeline và lưu kết quả vào một biến list để chuẩn bị nhồi vào YOLOv8
valid_sessions_list = filter_valid_sessions(OUTPUT_ROOT_DIR, MINIMUM_DURATION_MINUTES)

[PROCESS] Khởi chạy bộ lọc dữ liệu: Tìm kiếm các phiên quan sát >= 15.0 phút.
[PROCESS] Đang quét cấu trúc cây thư mục và phân tích Metadata...

[INFO] BÁO CÁO KẾT QUẢ SÀNG LỌC DỮ LIỆU ĐẦU VÀO CHO PHASE 1:
 - Tổng số phiên quan sát đã quét: 186
 - Số lượng phiên đạt tiêu chuẩn (>= 15.0 phút): 6
[INFO] DANH SÁCH DỮ LIỆU ƯU TIÊN (Sắp xếp theo thời lượng giảm dần):
  01. [26.98 phút] -> CAM_FY3407255/2025-07-16/091155_Total_26m59s
  02. [18.37 phút] -> CAM_F90136076/2025-07-17/084215_Total_18m22s
  03. [17.78 phút] -> CAM_FY3407255/2025-07-16/154509_Total_17m47s
  04. [17.33 phút] -> CAM_FY3407255/2025-07-15/195019_Total_17m20s
  05. [16.48 phút] -> CAM_FY3407255/2025-07-16/085503_Total_16m29s
  06. [15.08 phút] -> CAM_FY3407255/2025-07-15/193505_Total_15m5s
